In [ ]:
import json
import numpy as np
import geopandas as gpd


RESPONSE = "sepa"                               # "s1" or "sepa"
CLEAN_PATH = f"hex_clean_{RESPONSE}.gpkg"
Y_MODEL = f"log_{RESPONSE}_flood_frac"          # ternary was redundant; RESPONSE is already s1/sepa

# Load
gdf = gpd.read_file(CLEAN_PATH)
if gdf.crs is None or gdf.crs.to_epsg() != 3035:
    gdf = gdf.to_crs("EPSG:3035")

if Y_MODEL not in gdf.columns:
    raise KeyError(f"'{Y_MODEL}' not in {CLEAN_PATH}. Columns: {list(gdf.columns)}")

gdf = gdf.reset_index(drop=True)

In [21]:
gdf['greenspace_deficit'] = -gdf['greenspace_frac']
gdf['deprivation'] = -gdf['simd_rank']
gdf['sewer_deficit'] = -gdf['sewer_catchment_coverage']
gdf['flatter'] = -gdf['sqrt_mean_slope']
gdf['further_distance'] = -gdf['sqrt_dist_to_river_m']

In [22]:
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

direction_expectations = {
    # Exposure — all already point "high = hot / more exposed"
    "sqrt_pop_density": +1,
    "sqrt_building_footprint_frac": +1,
#sensitivity_vars
    "deprivation": +1,
    "age_dep_ratio": +1,
#adaptive_capacity_vars 
    "greenspace_deficit": +1,
    "sewer_deficit": +1,
# hazard control
    "sqrt_impervious_pct": +1,
    "flatter": +1,
    "further_distance": +1,
}

In [23]:
# Define the three vulnerability (HVI) domains and hazrds category

exposure_vars = [
    "sqrt_pop_density",
    "sqrt_building_footprint_frac",

]

sensitivity_vars = [
    "deprivation",
    "age_dep_ratio",
]

adaptive_capacity_vars = [
    "greenspace_deficit",
    "sewer_deficit",
]

hazard_control_vars= [
    "sqrt_impervious_pct",
    "flatter",
    "further_distance",
]

In [24]:
FULL_PATH = "hex_clean_sepa.gpkg"
import geopandas as gpd
print(gpd.read_file(FULL_PATH).columns.tolist())


['h3_id', 'hex_area_m2', 'land_area_m2', 'land_frac', 's1_flood_frac', 'sepa_flood_frac', 'sewer_catchment_coverage', 'mean_slope', 'impervious_pct', 'pop_density', 'age_dep_ratio', 'building_footprint_frac', 'simd_rank', 'greenspace_frac', 'dist_to_river_m', 'sqrt_pop_density', 'sqrt_building_footprint_frac', 'sqrt_impervious_pct', 'sqrt_mean_slope', 'sqrt_dist_to_river_m', 'log_sepa_flood_frac', 'no_population', 'sqrt_pop_density_z', 'simd_rank_z', 'age_dep_ratio_z', 'greenspace_frac_z', 'sewer_catchment_coverage_z', 'sqrt_impervious_pct_z', 'sqrt_mean_slope_z', 'sqrt_dist_to_river_m_z', 'geometry']


In [ ]:

import numpy as np
from pathlib import Path



RESPONSE = "sepa"                               # "s1" or "sepa"
CLEAN_PATH = f"hex_clean_{RESPONSE}.gpkg"
Y_MODEL = f"log_{RESPONSE}_flood_frac"          # ternary was redundant; RESPONSE is already s1/sepa
OUT_GPKG = Path(f"hex_clean_{RESPONSE}.gpkg")

# Load
gdf = gpd.read_file(CLEAN_PATH)
if gdf.crs is None or gdf.crs.to_epsg() != 3035:
    gdf = gdf.to_crs("EPSG:3035")

if Y_MODEL not in gdf.columns:
    raise KeyError(f"'{Y_MODEL}' not in {CLEAN_PATH}. Columns: {list(gdf.columns)}")

gdf = gdf.reset_index(drop=True)

id_col = "h3_id"

# Full input variable list across all three domains
all_input_vars = exposure_vars + sensitivity_vars + adaptive_capacity_vars + hazard_control_vars

# Load and intersection  
gdf = gpd.read_file(FULL_PATH)
if gdf.crs is None or gdf.crs.to_epsg() != 3035:
    gdf = gdf.to_crs("EPSG:3035")

# derive the sqrt_* features from their base columns
for v in all_input_vars:
    if v.startswith("sqrt_") and v not in gdf.columns:
        gdf[v] = np.sqrt(gdf[v[5:]].clip(lower=0))

both = (gdf[SEPA_FRAC] > 0) & (gdf[S1_FRAC] > 0)
inter = gdf[both].reset_index(drop=True)

n = len(inter)
print(f"Intersection sample: {n} hexes")
missing = [c for c in all_input_vars if c not in inter.columns]
if missing:
    raise KeyError(f"Missing predictors: {missing}")

# Confirm all variables are present and numeric in the GeoDataFrame
numeric_dtypes = ["float64", "int64", "float32", "int32"]
missing_vars = [v for v in all_input_vars if v not in gdf.columns]
non_numeric_vars = [v for v in all_input_vars if v in gdf.columns and gdf[v].dtype not in numeric_dtypes]

print(f"ID column:     {id_col}")
print(f"Total input variables: {len(all_input_vars)}\n")

print(f"  Exposure ({len(exposure_vars)}):           {exposure_vars}")
print(f"  Sensitivity ({len(sensitivity_vars)}):         {sensitivity_vars}")
print(f"  Adaptive capacity ({len(adaptive_capacity_vars)}):   {adaptive_capacity_vars}\n")
print(f"  Hazard control({len(hazard_control_vars)}):   {hazard_control_vars}\n")

Intersection sample: 550 hexes
ID column:     h3_id
Total input variables: 9

  Exposure (2):           ['sqrt_pop_density', 'sqrt_building_footprint_frac']
  Sensitivity (2):         ['simd_rank', 'age_dep_ratio']
  Adaptive capacity (2):   ['greenspace_frac', 'sewer_catchment_coverage']

  Hazard control(3):   ['sqrt_impervious_pct', 'sqrt_mean_slope', 'sqrt_dist_to_river_m']



#  OLS pipeline on the SEPA-and-S1 intersection subset.

Pipeline: transforms -> direction realignment -> subset to intersection ->
structural NaN impute -> listwise drop -> VIF report -> StandardScaler ->
KMO/Bartlett + 1-factor loadings -> OLS -> residual Moran's I -> save.

All predictors are realigned so that HIGH = MORE flood-vulnerable/hazardous
(direction_expectations == +1 for every predictor), so a coherent model should
return POSITIVE standardised coefficients; negatives are flagged.


In [ ]:
import pandas as pd
y = gdf["log_sepa_flood_frac"]
corr = pd.Series({v: gdf[v].corr(y) for v in all_input_vars}).sort_values()
print(corr.round(3))   

sqrt_building_footprint_frac   -0.239
sqrt_pop_density               -0.229
sqrt_impervious_pct            -0.209
age_dep_ratio                  -0.097
greenspace_deficit              0.035
deprivation                     0.063
flatter                         0.199
sewer_deficit                   0.212
further_distance                0.329
dtype: float64


# SEPA 

In [ ]:
#!/usr/bin/env python3


from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
from libpysal.weights import Queen
from esda.moran import Moran


RESPONSE   = "sepa"                     # response modelled: "sepa" or "s1"
HEX_PATH   = Path("hex_h3_res9_joined.gpkg")
TARGET_CRS = "EPSG:3035"
id_col     = "h3_id"

SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"
RESP_RAW  = {"s1": S1_FRAC, "sepa": SEPA_FRAC}[RESPONSE]
Y_MODEL   = f"log_{RESP_RAW}"
OUT_GPKG  = Path(f"hex_clean_{RESPONSE}_intersection.gpkg")
OUT_RPT   = Path(f"cleaning_report_{RESPONSE}_intersection.txt")

SQRT_VARS = ["pop_density", "building_footprint_frac", "impervious_pct",
             "mean_slope", "dist_to_river_m"]
VIF_THRESHOLD = 10.20

# Predictors removed from ALL models after the VIF report (collinearity).
DROP_AFTER_VIF = ["sqrt_building_footprint_frac"]

# Domains 
exposure_vars          = ["sqrt_pop_density", "sqrt_building_footprint_frac"]
sensitivity_vars       = ["deprivation", "age_dep_ratio"]
adaptive_capacity_vars = ["greenspace_deficit", "sewer_deficit"]
hazard_control_vars    = ["sqrt_impervious_pct", "flatter", "further_distance"]
all_input_vars = exposure_vars + sensitivity_vars + adaptive_capacity_vars + hazard_control_vars

direction_expectations = {
    "sqrt_pop_density": +1,
    "sqrt_building_footprint_frac": +1,
    "deprivation": +1,
    "age_dep_ratio": +1,
    "greenspace_deficit": +1,
    "sewer_deficit": +1,
    "sqrt_impervious_pct": +1,
    "flatter": +1,
    "further_distance": +1,
}

rpt = []
def log(m=""):
    print(m); rpt.append(str(m))

# Load
log("=" * 70)
log(f"OLS ON INTERSECTION - response = {RESPONSE} ({RESP_RAW})")
log("=" * 70)
gdf = gpd.read_file(HEX_PATH)
log(f"Loaded {len(gdf):,} hexes")
if gdf.crs is None or gdf.crs.to_epsg() != 3035:
    gdf = gdf.to_crs(TARGET_CRS); log(f"Reprojected to {TARGET_CRS}")

# Transforms and direction realignmnet (everything -> high = more vulnerable)
for v in SQRT_VARS:
    if v not in gdf.columns:
        raise KeyError(f"Raw predictor '{v}' missing; cannot build sqrt_{v}.")
    if (gdf[v] < 0).any():
        raise ValueError(f"{v} has negatives; sqrt undefined.")
    gdf[f"sqrt_{v}"] = np.sqrt(gdf[v])

gdf["deprivation"]       = -gdf["simd_rank"]                # rank 1 = most deprived
gdf["greenspace_deficit"] = -gdf["greenspace_frac"]
gdf["sewer_deficit"]      = -gdf["sewer_catchment_coverage"]
gdf["flatter"]            = -gdf["sqrt_mean_slope"]
gdf["further_distance"]   = -gdf["sqrt_dist_to_river_m"]

missing = [c for c in all_input_vars if c not in gdf.columns]
if missing:
    raise KeyError(f"Predictors missing after realignment: {missing}")

# Subset to intersection (SEPA>0 AND S1>0)
sepa_flooded = gdf[SEPA_FRAC] > 0
s1_flooded   = gdf[S1_FRAC]   > 0
intersection = sepa_flooded & s1_flooded
gdf = gdf[intersection].reset_index(drop=True)
log(f"\nIntersection subset (SEPA>0 & S1>0): {len(gdf):,} hexes")

# Structural missing values imputation  (age_dep_ratio on unpopulated hexes) + FLAG
if "age_dep_ratio" in gdf.columns and gdf["age_dep_ratio"].isna().any():
    n_missing = int(gdf["age_dep_ratio"].isna().sum())
    med = gdf["age_dep_ratio"].median()
    gdf["no_population"] = gdf["age_dep_ratio"].isna().astype(int)
    gdf["age_dep_ratio"] = gdf["age_dep_ratio"].fillna(med)
    log(f"age_dep_ratio: {n_missing:,} NaN -> median {med:.3f} (+ no_population flag)")

# Log response (safe: intersection guarantees RESP_RAW > 0)
gdf[Y_MODEL] = np.log(gdf[RESP_RAW])
log(f"{Y_MODEL}: raw skew {stats.skew(gdf[RESP_RAW]):+.2f} "
    f"-> log skew {stats.skew(gdf[Y_MODEL]):+.2f}")

# Listwise NaN DROP (Y + predictors)
need = [Y_MODEL] + all_input_vars
n1 = len(gdf)
gdf = gdf.dropna(subset=need).reset_index(drop=True)
log(f"Listwise NaN drop: {n1:,} -> {len(gdf):,}  |  MODELLING SAMPLE = {len(gdf):,}")
if len(gdf) < 50:
    log("[warning] very small sample; OLS may be unstable.")

# VIF
log(f"\n--- VIF (threshold {VIF_THRESHOLD}, report only) ---")
Xv = sm.add_constant(gdf[all_input_vars])
vif = pd.DataFrame({
    "variable": Xv.columns,
    "VIF": [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])],
})
for _, r in vif[vif["variable"] != "const"].sort_values("VIF", ascending=False).iterrows():
    flag = "  <-- > threshold" if r["VIF"] > VIF_THRESHOLD else ""
    log(f"    {r['variable']:<28}{r['VIF']:>7.2f}{flag}")

# Drop high-VIF predictors from EVERY downstream set --------------------
drop = [v for v in DROP_AFTER_VIF if v in all_input_vars]
if drop:
    all_input_vars         = [v for v in all_input_vars if v not in drop]
    exposure_vars          = [v for v in exposure_vars if v not in drop]
    sensitivity_vars       = [v for v in sensitivity_vars if v not in drop]
    adaptive_capacity_vars = [v for v in adaptive_capacity_vars if v not in drop]
    hazard_control_vars    = [v for v in hazard_control_vars if v not in drop]
    log(f"  DROPPED (collinearity): {drop}")
    log(f"  predictors remaining ({len(all_input_vars)}): {all_input_vars}")

    # re-check VIF on the reduced set so residual collinearity is visible
    Xv2 = sm.add_constant(gdf[all_input_vars])
    vif2 = pd.DataFrame({
        "variable": Xv2.columns,
        "VIF": [variance_inflation_factor(Xv2.values, i) for i in range(Xv2.shape[1])],
    })
    log("  VIF after drop:")
    for _, r in vif2[vif2["variable"] != "const"].sort_values("VIF", ascending=False).iterrows():
        flag = "  <-- still > threshold" if r["VIF"] > VIF_THRESHOLD else ""
        log(f"    {r['variable']:<28}{r['VIF']:>7.2f}{flag}")

# Stanbdarise (StandardScaler on the modelling subset -> _z)
scaler = StandardScaler()
Z = scaler.fit_transform(gdf[all_input_vars])
Z_COLS = [f"{c}_z" for c in all_input_vars]
gdf[Z_COLS] = Z
log("\nStandardised predictors with StandardScaler (fit on modelling subset).")

# KMO + BARTLETT + 1-FACTOR LOADINGS
log("\n--- Factorability (index-coherence check) ---")
Zdf = gdf[Z_COLS]
chi2, bart_p = calculate_bartlett_sphericity(Zdf)
kmo_per, kmo_total = calculate_kmo(Zdf)
log(f"  Bartlett chi2 = {chi2:.1f}, p = {bart_p:.4g}  (want p < 0.05)")
log(f"  KMO overall  = {kmo_total:.3f}  (want > 0.6)")

fa = FactorAnalyzer(n_factors=1, rotation=None)
fa.fit(Zdf)
load = fa.loadings_[:, 0]
log("  1-factor loadings (expect all +ve if directions are aligned):")
for c, l in zip(all_input_vars, load):
    exp = direction_expectations[c]
    ok = "OK" if np.sign(l) == np.sign(exp) else "UNEXPECTED SIGN"
    log(f"    {c:<28}{l:+.3f}   {ok}")

#  OLS  (log response ~ standardised realigned predictors)
log("\n--- OLS ---")
Xo = sm.add_constant(gdf[Z_COLS])
ols = sm.OLS(gdf[Y_MODEL].values, Xo.values).fit()
log(f"  R2 {ols.rsquared:.3f}  adj R2 {ols.rsquared_adj:.3f}  "
    f"AIC {ols.aic:.1f}  n {int(ols.nobs)}")
log(f"  {'predictor':<28}{'coef':>9}{'p':>9}   sign vs expected(+1)")
for name, coef, p in zip(["const"] + all_input_vars, ols.params, ols.pvalues):
    if name == "const":
        log(f"    {'const':<28}{coef:>9.3f}{p:>9.3g}")
        continue
    chk = "OK" if coef > 0 else "UNEXPECTED (-ve)"
    star = "*" if p < 0.05 else " "
    log(f"    {name:<28}{coef:>9.3f}{p:>9.3g} {star} {chk}")

# Residual  MORAN'S I  (full-model diagnostic; w reused below)
w = Queen.from_dataframe(gdf, use_index=False)
w.transform = "r"
mi = Moran(ols.resid, w)
log(f"\nMoran's I of full-model OLS residuals: {mi.I:.3f} (p = {mi.p_sim:.4f})")
log("  -> residual spatial autocorrelation present." if mi.p_sim < 0.05
    else "  -> no significant residual spatial autocorrelation.")

# AGGREGATED 4-CATEGORY OLS
#     domain score = mean of its standardised (+1-aligned) predictors,
#     re-standardised so the four coefficients are directly comparable.
log("\n--- Aggregated model: 4 domain scores ---")
domains = {
    "exposure_score":          exposure_vars,
    "sensitivity_score":       sensitivity_vars,
    "adaptive_capacity_score": adaptive_capacity_vars,
    "hazard_control_score":    hazard_control_vars,
}
for score, members in domains.items():
    gdf[score] = gdf[[f"{v}_z" for v in members]].mean(axis=1)

score_cols = list(domains.keys())
SCORE_Z = [f"{s}_z" for s in score_cols]
gdf[SCORE_Z] = StandardScaler().fit_transform(gdf[score_cols])

Xa = sm.add_constant(gdf[SCORE_Z])
ols_agg = sm.OLS(gdf[Y_MODEL].values, Xa.values).fit()
log(f"  R2 {ols_agg.rsquared:.3f}  adj R2 {ols_agg.rsquared_adj:.3f}  "
    f"AIC {ols_agg.aic:.1f}  n {int(ols_agg.nobs)}")
log(f"  {'domain':<28}{'coef':>9}{'p':>9}   sign vs expected(+1)")
for name, coef, p in zip(["const"] + score_cols, ols_agg.params, ols_agg.pvalues):
    if name == "const":
        log(f"    {'const':<28}{coef:>9.3f}{p:>9.3g}"); continue
    chk  = "OK" if coef > 0 else "UNEXPECTED (-ve)"
    star = "*" if p < 0.05 else " "
    log(f"    {name:<28}{coef:>9.3f}{p:>9.3g} {star} {chk}")

mi_agg = Moran(ols_agg.resid, w)
log(f"  Moran's I of aggregated-model residuals: {mi_agg.I:.3f} (p = {mi_agg.p_sim:.4f})")
log(f"  AIC comparison (lower = better): full {ols.aic:.1f} vs aggregated {ols_agg.aic:.1f}")

if "geometry" not in gdf.columns:
    base = gpd.read_file(HEX_PATH)[["h3_id", "geometry"]]
    gdf = gdf.merge(base, on="h3_id", how="left")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=TARGET_CRS)
# Save
gdf.to_file(OUT_GPKG, driver="GPKG")
json.dump(
    {"full_predictors": all_input_vars, "aggregated_predictors": score_cols},
    open(f"hex_clean_{RESPONSE}_intersection_predictors.json", "w"),
)
Path(OUT_RPT).write_text("\n".join(rpt))
log(f"\nSaved: {OUT_GPKG}  |  {OUT_RPT}")

OLS ON INTERSECTION - response = sepa (sepa_flood_frac)
Loaded 4,201 hexes

Intersection subset (SEPA>0 & S1>0): 550 hexes
age_dep_ratio: 126 NaN -> median 0.297 (+ no_population flag)
log_sepa_flood_frac: raw skew +3.82 -> log skew -1.20
Listwise NaN drop: 550 -> 537  |  MODELLING SAMPLE = 537

--- VIF (threshold 10.2, report only) ---
    sqrt_building_footprint_frac  13.01  <-- > threshold
    sqrt_impervious_pct           10.16
    sqrt_pop_density               6.74
    sewer_deficit                  5.93
    deprivation                    1.27
    age_dep_ratio                  1.13
    further_distance               1.08
    greenspace_deficit             1.03
    flatter                        1.02
  DROPPED (collinearity): ['sqrt_building_footprint_frac']
  predictors remaining (8): ['sqrt_pop_density', 'deprivation', 'age_dep_ratio', 'greenspace_deficit', 'sewer_deficit', 'sqrt_impervious_pct', 'flatter', 'further_distance']
  VIF after drop:
    sewer_deficit                

/opt/anaconda3/envs/spatial-regression/lib/python3.14/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 145 disconnected components.
 There are 75 islands with ids: 23, 24, 50, 51, 52, 58, 61, 64, 101, 105, 108, 112, 113, 123, 137, 145, 152, 155, 161, 162, 165, 181, 192, 197, 205, 210, 226, 228, 230, 235, 236, 277, 279, 280, 282, 284, 289, 293, 302, 308, 309, 319, 323, 324, 337, 345, 351, 356, 357, 362, 366, 368, 371, 375, 378, 398, 399, 404, 411, 419, 436, 442, 443, 451, 454, 460, 461, 464, 466, 487, 489, 500, 512, 523, 533.
  W.__init__(self, neighbors, ids=ids, **kw)


In [49]:
OUT_GPKG  = Path(f"hex_clean_{RESPONSE}_intersection.gpkg")
import geopandas as gpd
print(gpd.read_file(OUT_GPKG).columns.tolist())

['h3_id', 'hex_area_m2', 'land_area_m2', 'land_frac', 's1_flood_frac', 'sepa_flood_frac', 'sewer_catchment_coverage', 'mean_slope', 'impervious_pct', 'pop_density', 'age_dep_ratio', 'building_footprint_frac', 'simd_rank', 'greenspace_frac', 'dist_to_river_m', 'sqrt_pop_density', 'sqrt_building_footprint_frac', 'sqrt_impervious_pct', 'sqrt_mean_slope', 'sqrt_dist_to_river_m', 'deprivation', 'greenspace_deficit', 'sewer_deficit', 'flatter', 'further_distance', 'no_population', 'log_s1_flood_frac', 'sqrt_pop_density_z', 'sqrt_building_footprint_frac_z', 'deprivation_z', 'age_dep_ratio_z', 'greenspace_deficit_z', 'sewer_deficit_z', 'sqrt_impervious_pct_z', 'flatter_z', 'further_distance_z', 'geometry']


# Sentinel-1 

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
from libpysal.weights import Queen
from esda.moran import Moran


RESPONSE   = "s1"                     # response modelled: "sepa" or "s1"
HEX_PATH   = Path("hex_h3_res9_joined.gpkg")
TARGET_CRS = "EPSG:3035"
id_col     = "h3_id"

SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"
RESP_RAW  = {"s1": S1_FRAC, "sepa": SEPA_FRAC}[RESPONSE]
Y_MODEL   = f"log_{RESP_RAW}"
OUT_GPKG  = Path(f"hex_clean_{RESPONSE}_intersection.gpkg")
OUT_RPT   = Path(f"cleaning_report_{RESPONSE}_intersection.txt")

SQRT_VARS = ["pop_density", "building_footprint_frac", "impervious_pct",
             "mean_slope", "dist_to_river_m"]
VIF_THRESHOLD = 10.20

# Predictors removed from ALL models after the VIF report (collinearity).
DROP_AFTER_VIF = ["sqrt_building_footprint_frac"]

# Domains 
exposure_vars          = ["sqrt_pop_density", "sqrt_building_footprint_frac"]
sensitivity_vars       = ["deprivation", "age_dep_ratio"]
adaptive_capacity_vars = ["greenspace_deficit", "sewer_deficit"]
hazard_control_vars    = ["sqrt_impervious_pct", "flatter", "further_distance"]
all_input_vars = exposure_vars + sensitivity_vars + adaptive_capacity_vars + hazard_control_vars

direction_expectations = {
    "sqrt_pop_density": +1,
    "sqrt_building_footprint_frac": +1,
    "deprivation": +1,
    "age_dep_ratio": +1,
    "greenspace_deficit": +1,
    "sewer_deficit": +1,
    "sqrt_impervious_pct": +1,
    "flatter": +1,
    "further_distance": +1,
}

rpt = []
def log(m=""):
    print(m); rpt.append(str(m))

# Load
log("=" * 70)
log(f"OLS ON INTERSECTION - response = {RESPONSE} ({RESP_RAW})")
log("=" * 70)
gdf = gpd.read_file(HEX_PATH)
log(f"Loaded {len(gdf):,} hexes")
if gdf.crs is None or gdf.crs.to_epsg() != 3035:
    gdf = gdf.to_crs(TARGET_CRS); log(f"Reprojected to {TARGET_CRS}")

# Trasformations and Direction Realignment (everything -> high = more vulnerable)
for v in SQRT_VARS:
    if v not in gdf.columns:
        raise KeyError(f"Raw predictor '{v}' missing; cannot build sqrt_{v}.")
    if (gdf[v] < 0).any():
        raise ValueError(f"{v} has negatives; sqrt undefined.")
    gdf[f"sqrt_{v}"] = np.sqrt(gdf[v])

gdf["deprivation"]       = -gdf["simd_rank"]                # rank 1 = most deprived
gdf["greenspace_deficit"] = -gdf["greenspace_frac"]
gdf["sewer_deficit"]      = -gdf["sewer_catchment_coverage"]
gdf["flatter"]            = -gdf["sqrt_mean_slope"]
gdf["further_distance"]   = -gdf["sqrt_dist_to_river_m"]

missing = [c for c in all_input_vars if c not in gdf.columns]
if missing:
    raise KeyError(f"Predictors missing after realignment: {missing}")

# Subset to intresection (SEPA>0 AND S1>0)
sepa_flooded = gdf[SEPA_FRAC] > 0
s1_flooded   = gdf[S1_FRAC]   > 0
intersection = sepa_flooded & s1_flooded
gdf = gdf[intersection].reset_index(drop=True)
log(f"\nIntersection subset (SEPA>0 & S1>0): {len(gdf):,} hexes")

# Structutral missinf imputaiton (age_dep_ratio on unpopulated hexes) + FLAG
if "age_dep_ratio" in gdf.columns and gdf["age_dep_ratio"].isna().any():
    n_missing = int(gdf["age_dep_ratio"].isna().sum())
    med = gdf["age_dep_ratio"].median()
    gdf["no_population"] = gdf["age_dep_ratio"].isna().astype(int)
    gdf["age_dep_ratio"] = gdf["age_dep_ratio"].fillna(med)
    log(f"age_dep_ratio: {n_missing:,} NaN -> median {med:.3f} (+ no_population flag)")

# Log Response (safe: intersection guarantees RESP_RAW > 0)
gdf[Y_MODEL] = np.log(gdf[RESP_RAW])
log(f"{Y_MODEL}: raw skew {stats.skew(gdf[RESP_RAW]):+.2f} "
    f"-> log skew {stats.skew(gdf[Y_MODEL]):+.2f}")

# Likewise NaN DROP (Y + predictors)
need = [Y_MODEL] + all_input_vars
n1 = len(gdf)
gdf = gdf.dropna(subset=need).reset_index(drop=True)
log(f"Listwise NaN drop: {n1:,} -> {len(gdf):,}  |  MODELLING SAMPLE = {len(gdf):,}")
if len(gdf) < 50:
    log("[warning] very small sample; OLS may be unstable.")

# VIF 
log(f"\n--- VIF (threshold {VIF_THRESHOLD}, report only) ---")
Xv = sm.add_constant(gdf[all_input_vars])
vif = pd.DataFrame({
    "variable": Xv.columns,
    "VIF": [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])],
})
for _, r in vif[vif["variable"] != "const"].sort_values("VIF", ascending=False).iterrows():
    flag = "  <-- > threshold" if r["VIF"] > VIF_THRESHOLD else ""
    log(f"    {r['variable']:<28}{r['VIF']:>7.2f}{flag}")

# Drop high-VIF predictors from EVERY downstream set 
drop = [v for v in DROP_AFTER_VIF if v in all_input_vars]
if drop:
    all_input_vars         = [v for v in all_input_vars if v not in drop]
    exposure_vars          = [v for v in exposure_vars if v not in drop]
    sensitivity_vars       = [v for v in sensitivity_vars if v not in drop]
    adaptive_capacity_vars = [v for v in adaptive_capacity_vars if v not in drop]
    hazard_control_vars    = [v for v in hazard_control_vars if v not in drop]
    log(f"  DROPPED (collinearity): {drop}")
    log(f"  predictors remaining ({len(all_input_vars)}): {all_input_vars}")

    # re-check VIF on the reduced set so residual collinearity is visible
    Xv2 = sm.add_constant(gdf[all_input_vars])
    vif2 = pd.DataFrame({
        "variable": Xv2.columns,
        "VIF": [variance_inflation_factor(Xv2.values, i) for i in range(Xv2.shape[1])],
    })
    log("  VIF after drop:")
    for _, r in vif2[vif2["variable"] != "const"].sort_values("VIF", ascending=False).iterrows():
        flag = "  <-- still > threshold" if r["VIF"] > VIF_THRESHOLD else ""
        log(f"    {r['variable']:<28}{r['VIF']:>7.2f}{flag}")

# Standarise (StandardScaler on the modelling subset -> _z)
scaler = StandardScaler()
Z = scaler.fit_transform(gdf[all_input_vars])
Z_COLS = [f"{c}_z" for c in all_input_vars]
gdf[Z_COLS] = Z
log("\nStandardised predictors with StandardScaler (fit on modelling subset).")

# KMO + BARTLETT + 1-FACTOR LOADINGS
log("\n--- Factorability (index-coherence check) ---")
Zdf = gdf[Z_COLS]
chi2, bart_p = calculate_bartlett_sphericity(Zdf)
kmo_per, kmo_total = calculate_kmo(Zdf)
log(f"  Bartlett chi2 = {chi2:.1f}, p = {bart_p:.4g}  (want p < 0.05)")
log(f"  KMO overall  = {kmo_total:.3f}  (want > 0.6)")

fa = FactorAnalyzer(n_factors=1, rotation=None)
fa.fit(Zdf)
load = fa.loadings_[:, 0]
log("  1-factor loadings (expect all +ve if directions are aligned):")
for c, l in zip(all_input_vars, load):
    exp = direction_expectations[c]
    ok = "OK" if np.sign(l) == np.sign(exp) else "UNEXPECTED SIGN"
    log(f"    {c:<28}{l:+.3f}   {ok}")

# OLS  (log response ~ standardised realigned predictors)
log("\n--- OLS ---")
Xo = sm.add_constant(gdf[Z_COLS])
ols = sm.OLS(gdf[Y_MODEL].values, Xo.values).fit()
log(f"  R2 {ols.rsquared:.3f}  adj R2 {ols.rsquared_adj:.3f}  "
    f"AIC {ols.aic:.1f}  n {int(ols.nobs)}")
log(f"  {'predictor':<28}{'coef':>9}{'p':>9}   sign vs expected(+1)")
for name, coef, p in zip(["const"] + all_input_vars, ols.params, ols.pvalues):
    if name == "const":
        log(f"    {'const':<28}{coef:>9.3f}{p:>9.3g}")
        continue
    chk = "OK" if coef > 0 else "UNEXPECTED (-ve)"
    star = "*" if p < 0.05 else " "
    log(f"    {name:<28}{coef:>9.3f}{p:>9.3g} {star} {chk}")

# Residual MORAN'S I  (full-model diagnostic; w reused below)
w = Queen.from_dataframe(gdf, use_index=False)
w.transform = "r"
mi = Moran(ols.resid, w)
log(f"\nMoran's I of full-model OLS residuals: {mi.I:.3f} (p = {mi.p_sim:.4f})")
log("  -> residual spatial autocorrelation present." if mi.p_sim < 0.05
    else "  -> no significant residual spatial autocorrelation.")

# Agreeagte 4-Ccategory OLS
#     domain score = mean of its standardised (+1-aligned) predictors,
#     re-standardised so the four coefficients are directly comparable.
log("\n--- Aggregated model: 4 domain scores ---")
domains = {
    "exposure_score":          exposure_vars,
    "sensitivity_score":       sensitivity_vars,
    "adaptive_capacity_score": adaptive_capacity_vars,
    "hazard_control_score":    hazard_control_vars,
}
for score, members in domains.items():
    gdf[score] = gdf[[f"{v}_z" for v in members]].mean(axis=1)

score_cols = list(domains.keys())
SCORE_Z = [f"{s}_z" for s in score_cols]
gdf[SCORE_Z] = StandardScaler().fit_transform(gdf[score_cols])

Xa = sm.add_constant(gdf[SCORE_Z])
ols_agg = sm.OLS(gdf[Y_MODEL].values, Xa.values).fit()
log(f"  R2 {ols_agg.rsquared:.3f}  adj R2 {ols_agg.rsquared_adj:.3f}  "
    f"AIC {ols_agg.aic:.1f}  n {int(ols_agg.nobs)}")
log(f"  {'domain':<28}{'coef':>9}{'p':>9}   sign vs expected(+1)")
for name, coef, p in zip(["const"] + score_cols, ols_agg.params, ols_agg.pvalues):
    if name == "const":
        log(f"    {'const':<28}{coef:>9.3f}{p:>9.3g}"); continue
    chk  = "OK" if coef > 0 else "UNEXPECTED (-ve)"
    star = "*" if p < 0.05 else " "
    log(f"    {name:<28}{coef:>9.3f}{p:>9.3g} {star} {chk}")

mi_agg = Moran(ols_agg.resid, w)
log(f"  Moran's I of aggregated-model residuals: {mi_agg.I:.3f} (p = {mi_agg.p_sim:.4f})")
log(f"  AIC comparison (lower = better): full {ols.aic:.1f} vs aggregated {ols_agg.aic:.1f}")

if "geometry" not in gdf.columns:
    base = gpd.read_file(HEX_PATH)[["h3_id", "geometry"]]
    gdf = gdf.merge(base, on="h3_id", how="left")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=TARGET_CRS)
#Save 
gdf.to_file(OUT_GPKG, driver="GPKG")
json.dump(
    {"full_predictors": all_input_vars, "aggregated_predictors": score_cols},
    open(f"hex_clean_{RESPONSE}_intersection_predictors.json", "w"),
)
Path(OUT_RPT).write_text("\n".join(rpt))
log(f"\nSaved: {OUT_GPKG}  |  {OUT_RPT}")

OLS ON INTERSECTION - response = s1 (s1_flood_frac)
Loaded 4,201 hexes

Intersection subset (SEPA>0 & S1>0): 550 hexes
age_dep_ratio: 126 NaN -> median 0.297 (+ no_population flag)
log_s1_flood_frac: raw skew +5.79 -> log skew -0.93
Listwise NaN drop: 550 -> 537  |  MODELLING SAMPLE = 537

--- VIF (threshold 10.2, report only) ---
    sqrt_building_footprint_frac  13.01  <-- > threshold
    sqrt_impervious_pct           10.16
    sqrt_pop_density               6.74
    sewer_deficit                  5.93
    deprivation                    1.27
    age_dep_ratio                  1.13
    further_distance               1.08
    greenspace_deficit             1.03
    flatter                        1.02
  DROPPED (collinearity): ['sqrt_building_footprint_frac']
  predictors remaining (8): ['sqrt_pop_density', 'deprivation', 'age_dep_ratio', 'greenspace_deficit', 'sewer_deficit', 'sqrt_impervious_pct', 'flatter', 'further_distance']
  VIF after drop:
    sewer_deficit                  5.89

/opt/anaconda3/envs/spatial-regression/lib/python3.14/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 145 disconnected components.
 There are 75 islands with ids: 23, 24, 50, 51, 52, 58, 61, 64, 101, 105, 108, 112, 113, 123, 137, 145, 152, 155, 161, 162, 165, 181, 192, 197, 205, 210, 226, 228, 230, 235, 236, 277, 279, 280, 282, 284, 289, 293, 302, 308, 309, 319, 323, 324, 337, 345, 351, 356, 357, 362, 366, 368, 371, 375, 378, 398, 399, 404, 411, 419, 436, 442, 443, 451, 454, 460, 461, 464, 466, 487, 489, 500, 512, 523, 533.
  W.__init__(self, neighbors, ids=ids, **kw)


# Spatially-blocked bootstrap test of representational injustice (H2).

Two global linear models are fitted on the intersection sample (hexes flooded in
BOTH Sentinel-1 and SEPA), with identical, standardised predictors:
    model_S1  : log(S1 flood intensity)   ~ standardised predictors
    model_SEPA: log(SEPA flood intensity) ~ standardised predictors
For each predictor k the coefficient difference d_k = beta_S1,k - beta_SEPA,k is
tested against zero with a spatially-blocked bootstrap (Lahiri 2003; Harris 2017),
resampling contiguous spatial blocks to preserve local spatial dependence.

Responses are left on their native (log) SCALE - they are NOT z-scored. Predictors
ARE standardised (so a coefficient = effect per SD of predictor, comparable across
predictors). 

Interpretation:
  d_k significantly > 0  -> SEPA UNDER-weights predictor k vs Sentinel-1
  d_k significantly < 0  -> SEPA OVER-weights predictor k vs Sentinel-1
  either sign, if significant AND non-trivial in size -> representational injustice
  small / non-significant -> the two representations are consistent


In [ ]:
from pathlib import Path
import numpy as np
import geopandas as gpd


SEPA_PATH = Path("hex_clean_sepa_intersection_simplepredictors.gpkg")
S1_PATH   = Path("hex_clean_s1_intersection_simplepredictors.gpkg")
TARGET_CRS = "EPSG:3035"

PREDICTORS = [
    "sqrt_pop_density",
    "simd_rank",
    "age_dep_ratio",
    "greenspace_frac",
    "sewer_catchment_coverage",
    "sqrt_impervious_pct",
    "sqrt_mean_slope",
    "sqrt_dist_to_river_m",
]

PRIMARY_EJ = "age_dep_ratio"
BLOCK_HEXWIDTHS = 5
HEX_WIDTH_M = 300.0
N_BOOT = 1000
SEED = 42
rng = np.random.default_rng(SEED)

# Load
g_sepa = gpd.read_file(SEPA_PATH)
g_s1 = gpd.read_file(S1_PATH)

if g_sepa.crs is None or g_sepa.crs.to_epsg() != 3035:
    g_sepa = g_sepa.to_crs(TARGET_CRS)
if g_s1.crs is None or g_s1.crs.to_epsg() != 3035:
    g_s1 = g_s1.to_crs(TARGET_CRS)

if len(g_sepa) != len(g_s1):
    raise ValueError(f"Row counts differ: SEPA={len(g_sepa)}, S1={len(g_s1)}")

# Align + merge safely 
key_cols = [c for c in ["hex_id", "h3_id", "index", "id"] if c in g_sepa.columns and c in g_s1.columns]

if key_cols:
    key = key_cols[0]
    geom_col = g_sepa.geometry.name

    s1_attrs = g_s1.drop(columns=[g_s1.geometry.name], errors="ignore").copy()

    dup_cols = [c for c in s1_attrs.columns if c != key and c in g_sepa.columns]
    s1_attrs = s1_attrs.rename(columns={c: f"{c}_s1src" for c in dup_cols})

    g = g_sepa.merge(s1_attrs, on=key, how="inner")
    g = gpd.GeoDataFrame(g, geometry=geom_col, crs=g_sepa.crs)

else:
    g = g_sepa.copy()
    s1_attrs = g_s1.drop(columns=[g_s1.geometry.name], errors="ignore").copy()

    for c in s1_attrs.columns:
        if c not in g.columns:
            g[c] = s1_attrs[c].values
        else:
            g[f"{c}_s1src"] = s1_attrs[c].values

# Identify response columns 
sepa_candidates = ["sepa_flood_frac", "log_sepa_flood_frac"]
s1_candidates   = ["s1_flood_frac", "log_s1_flood_frac"]

sepa_col = next((c for c in sepa_candidates if c in g.columns), None)
s1_col   = next((c for c in s1_candidates if c in g.columns), None)

if sepa_col is None:
    raise KeyError(f"No SEPA response column found. Tried: {sepa_candidates}")
if s1_col is None:
    raise KeyError(f"No S1 response column found. Tried: {s1_candidates}")

# Intersection sample 
if "sepa_flood_frac" in g.columns and "s1_flood_frac" in g.columns:
    inter = g[(g["sepa_flood_frac"] > 0) & (g["s1_flood_frac"] > 0)].reset_index(drop=True)
else:
    inter = g.reset_index(drop=True)

n = len(inter)
print(f"Intersection sample: {n} hexes")

missing = [c for c in PREDICTORS if c not in inter.columns]
if missing:
    raise KeyError(f"Missing predictors: {missing}")

# Responses on native log scale 
if s1_col == "s1_flood_frac":
    yS1 = np.log(inter[s1_col].to_numpy())
else:
    yS1 = inter[s1_col].to_numpy()

if sepa_col == "sepa_flood_frac":
    ySEPA = np.log(inter[sepa_col].to_numpy())
else:
    ySEPA = inter[sepa_col].to_numpy()

# Standardised predictors (+ intercept) 
Xraw = inter[PREDICTORS].to_numpy(dtype=float)
Xz = (Xraw - Xraw.mean(axis=0)) / Xraw.std(axis=0, ddof=0)
X = np.column_stack([np.ones(n), Xz])

names = ["Intercept"] + PREDICTORS
k = X.shape[1]

def ols_beta(Xm, ym):
    beta, *_ = np.linalg.lstsq(Xm, ym, rcond=None)
    return beta

bS1_full = ols_beta(X, yS1)
bSEPA_full = ols_beta(X, ySEPA)
diff_full = bS1_full - bSEPA_full

# Spatial blocks 
cent = inter.geometry.centroid
xy = np.column_stack([cent.x.to_numpy(), cent.y.to_numpy()])

block_size = BLOCK_HEXWIDTHS * HEX_WIDTH_M
bx = np.floor((xy[:, 0] - xy[:, 0].min()) / block_size).astype(int)
by = np.floor((xy[:, 1] - xy[:, 1].min()) / block_size).astype(int)
block_id = bx * (by.max() + 1) + by

blocks = [np.where(block_id == b)[0] for b in np.unique(block_id)]
n_blocks = len(blocks)

print(
    f"Spatial blocks: {n_blocks} ({block_size:.0f} m = {BLOCK_HEXWIDTHS} hex widths); "
    f"mean {n / n_blocks:.1f} hexes/block"
)

# Spatially-blocked bootstrap
boot_diff = np.full((N_BOOT, k), np.nan)

for b in range(N_BOOT):
    parts = []
    tot = 0
    while tot < n:
        blk = blocks[rng.integers(n_blocks)]
        parts.append(blk)
        tot += len(blk)

    idx = np.concatenate(parts)[:n]
    Xb = X[idx]

    try:
        boot_diff[b] = ols_beta(Xb, yS1[idx]) - ols_beta(Xb, ySEPA[idx])
    except Exception:
        continue

boot_diff = boot_diff[~np.isnan(boot_diff).any(axis=1)]
print(f"Successful replicates: {len(boot_diff)} / {N_BOOT}")

# Inference
lo = np.percentile(boot_diff, 2.5, axis=0)
hi = np.percentile(boot_diff, 97.5, axis=0)

p_gt = (boot_diff > 0).mean(axis=0)
p_two = np.clip(2 * np.minimum(p_gt, 1 - p_gt), 1 / len(boot_diff), 1.0)

def material(i):
    if p_two[i] >= 0.05:
        return False
    ref = abs(bS1_full[i]) if abs(bS1_full[i]) > 1e-6 else 1.0
    return abs(diff_full[i]) / ref >= 0.25

print("\n" + "=" * 100)
print(f"{'Predictor':<26}{'b_S1':>8}{'b_SEPA':>9}{'diff':>8}{'95% CI':>20}{'p':>8}  interpretation")
print("-" * 100)

for i, nm in enumerate(names):
    if nm == "Intercept":
        continue

    d = diff_full[i]
    sig = p_two[i] < 0.05
    mat = material(i)

    if not sig:
        interp = "consistent (no injustice)"
    elif not mat:
        interp = "sig but trivial in size (treat as consistent)"
    elif d > 0:
        interp = "SEPA UNDER-weights -> injustice"
    else:
        interp = "SEPA OVER-weights -> injustice"

    star = "*" if sig else " "
    print(
        f"{nm:<26}{bS1_full[i]:>8.3f}{bSEPA_full[i]:>9.3f}{d:>8.3f}"
        f"  [{lo[i]:+.3f}, {hi[i]:+.3f}]{p_two[i]:>8.3f}{star} {interp}"
    )

print("=" * 100)
print("* p<0.05 (blocked bootstrap). 'Material' = significant AND |diff| >= 25% of |b_S1|.")

Intersection sample: 537 hexes
Spatial blocks: 132 (1500 m = 5 hex widths); mean 4.1 hexes/block
Successful replicates: 1000 / 1000

Predictor                     b_S1   b_SEPA    diff              95% CI       p  interpretation
----------------------------------------------------------------------------------------------------
sqrt_pop_density             0.089   -0.377   0.467  [-0.088, +1.151]   0.106  consistent (no injustice)
simd_rank                   -0.100   -0.209   0.110  [-0.162, +0.361]   0.420  consistent (no injustice)
age_dep_ratio               -0.182   -0.185   0.003  [-0.196, +0.193]   0.936  consistent (no injustice)
greenspace_frac             -0.087   -0.070  -0.017  [-0.236, +0.199]   0.892  consistent (no injustice)
sewer_catchment_coverage    -0.460   -0.036  -0.424  [-0.964, +0.026]   0.072  consistent (no injustice)
sqrt_impervious_pct          0.722   -0.178   0.900  [+0.428, +1.300]   0.002* SEPA UNDER-weights -> injustice
sqrt_mean_slope             -0.287